# GPU runner

For the phases that need CUDA: comparator reproduction, large open-weight
compressors, optional open-weight receiver robustness, and larger sweeps.

**Rules this notebook follows.**

- No secret is embedded in the notebook JSON. Credentials come from the
  runtime's secret store or the environment.
- No experimental phase runs automatically. Every phase is behind an explicit
  flag that is `False` by default.
- Results are written to a configured persistent location and exported for
  local aggregation; nothing is left only in the ephemeral runtime.
- No machine-specific path appears anywhere in this file.


## 1. Environment


In [ ]:
import os, subprocess, sys
print(sys.version)
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no nvidia-smi: this runtime has no GPU')


## 2. Install

Point `REPO_URL` at the public repository, or upload a source archive to the
runtime and install from the extracted directory.


In [ ]:
REPO_URL = os.environ.get('HANDOFF_REPO_URL', '')  # set this, or install from an upload
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)

if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, f'{WORK}/repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{WORK}/repo[dev,ml,figures,data]'], check=True)
else:
    print('Set HANDOFF_REPO_URL, or upload and extract a source archive into', WORK)


## 3. Credentials

Read from the runtime's secret store if present, otherwise from the
environment. Values are never printed and never written into a cell.


In [ ]:
def load_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
    except Exception:
        value = None
    return value or os.environ.get(name, '')

for key in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'HF_TOKEN'):
    value = load_secret(key)
    if value:
        os.environ[key] = value
    print(f'{key}: {"set" if value else "unset"}')  # presence only, never the value


## 4. Persistent result storage

Mount or configure durable storage so results survive the runtime. The
location is supplied at run time; nothing is hard-coded.


In [ ]:
RESULTS_ROOT = os.environ.get('HANDOFF_RESULTS_ROOT', f'{WORK}/results')
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.environ['HANDOFF_PRIVATE_HOME'] = RESULTS_ROOT
print('results root configured')

# To use Drive instead:
# from google.colab import drive; drive.mount('/content/drive')
# RESULTS_ROOT = '/content/drive/MyDrive/<your-folder>'


## 5. Health checks

These make no model call.


In [ ]:
subprocess.run(['handoff', 'doctor'], check=False)
subprocess.run(['handoff', 'self-test'], check=True)
subprocess.run(['handoff', 'stages'], check=False)
subprocess.run(['handoff', 'baselines'], check=False)


## 6. Phase selection

Every flag is `False`. Turn on exactly one phase at a time, and only after
the corresponding freeze record validates.


In [ ]:
RUN_BASELINE_REPRODUCTION = False   # public/native tasks only, never the sealed test split
RUN_CALIBRATION           = False   # instrument validation on the calibration pool
RUN_STAGE1                = False   # requires the protocol freeze record
RUN_STAGE2_DEV            = False   # requires the protocol freeze record and a passed gate
RUN_STAGE2_TEST           = False   # requires the FINAL TEST freeze record

enabled = [k for k, v in globals().items() if k.startswith('RUN_') and v]
print('enabled phases:', enabled or 'none')
assert len(enabled) <= 1, 'run one phase at a time'


## 7. Comparator reproduction (GPU)

Clone official implementations into the private third-party area, pin exact
revisions in the registry, and reproduce a published result on a public task.

A method that fails is recorded as unreproduced. It is not repaired after
looking at the benchmark, and not replaced by an easier one.


In [ ]:
if RUN_BASELINE_REPRODUCTION:
    raise NotImplementedError(
        'Bind the official implementations first: clone into '
        f'{RESULTS_ROOT}/third_party/<method>, pin the commit and checkpoint in '
        'configs/baselines.yaml, then run the reproduction gate.'
    )
else:
    print('skipped')


## 8. Freeze verification

No experimental phase may run before this passes.


In [ ]:
FREEZE_RECORD = os.environ.get('HANDOFF_FREEZE_RECORD', '')
if FREEZE_RECORD:
    subprocess.run(['handoff', 'freeze-verify', '--record', FREEZE_RECORD], check=False)
else:
    print('no freeze record configured: experimental phases remain blocked')


## 9. Export

Package results for local aggregation. Outputs only; no credential, no raw
source.


In [ ]:
import shutil, datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive = f'{WORK}/handoff-results-{stamp}'
for sub in ('runs', 'tables', 'figures', 'artifacts'):
    os.makedirs(f'{RESULTS_ROOT}/{sub}', exist_ok=True)
shutil.make_archive(archive, 'zip', RESULTS_ROOT, base_dir='.')
print('wrote', archive + '.zip')
print('Download it, then aggregate locally with the frozen analysis scripts.')
